In [1]:
# Phase 5 – Network Analysis
# Cell 1: Setup + Load Temporal Networks from Phase 4

from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import networkx as nx
import pickle

# ---------------------- Paths ----------------------
PROJECT = Path(r"D:\univercity\omid_project\economics_project")

PHASE4_DATA = PROJECT / "outputs" / "phase4" / "data"
OUT_DIR     = PROJECT / "outputs" / "phase5"

OUT_DATA  = OUT_DIR / "data"
OUT_TAB   = OUT_DIR / "tables"
OUT_FIG   = OUT_DIR / "figures"
OUT_LOG   = OUT_DIR / "logs"

for d in [OUT_DATA, OUT_TAB, OUT_FIG, OUT_LOG]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Phase 5 – Network Analysis | Cell 1")
print("=" * 60)
print(f"Timestamp     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Project Root  : {PROJECT}")
print(f"Outputs       : {OUT_DIR}")
print()

# Load cleaned temporal networks
with open(PHASE4_DATA / "temporal_networks_clean.pkl", "rb") as f:
    networks = pickle.load(f)

print(f"Loaded networks: {len(networks)}")
print(f"Sample keys   : {list(networks.keys())[:5]}")
print("\nCell 1 completed successfully.")

Phase 5 – Network Analysis | Cell 1
Timestamp     : 2026-09-23 10:51:06
Project Root  : D:\univercity\omid_project\economics_project
Outputs       : D:\univercity\omid_project\economics_project\outputs\phase5

Loaded networks: 2718
Sample keys   : [('Australia', 1870), ('Australia', 1871), ('Australia', 1872), ('Australia', 1873), ('Australia', 1874)]

Cell 1 completed successfully.


In [2]:
# Phase 5 – Network Analysis
# Cell 2: Centrality Measures

import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path
import pickle

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase5" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase5" / "tables"

with open(PROJECT / "outputs" / "phase4" / "data" / "temporal_networks_clean.pkl", "rb") as f:
    networks = pickle.load(f)

print("=" * 60)
print("Phase 5 – Network Analysis | Cell 2")
print("Computing Centrality Measures")
print("=" * 60)

records = []

for (country, year), G in networks.items():
    # Ensure weights are valid
    for u, v, d in G.edges(data=True):
        if d.get("weight", 0) <= 0 or np.isnan(d.get("weight", 0)):
            d["weight"] = 1e-6

    try:
        # Degree Centrality (weighted strength)
        in_strength  = dict(G.in_degree(weight="weight"))
        out_strength = dict(G.out_degree(weight="weight"))
        
        # Betweenness
        betweenness = nx.betweenness_centrality(G, weight="weight", normalized=True)
        
        # Closeness
        closeness = nx.closeness_centrality(G, distance="weight")
        
        # Eigenvector
        try:
            eigenvector = nx.eigenvector_centrality_numpy(G, weight="weight")
        except:
            eigenvector = {n: 0.0 for n in G.nodes()}
        
        # PageRank
        pagerank = nx.pagerank(G, weight="weight", max_iter=200)
        
        # Katz
        try:
            katz = nx.katz_centrality_numpy(G, weight="weight", alpha=0.1)
        except:
            katz = {n: 0.0 for n in G.nodes()}
        
        for node in G.nodes():
            records.append({
                "country": country,
                "year": year,
                "node": node,
                "in_strength": in_strength.get(node, 0),
                "out_strength": out_strength.get(node, 0),
                "betweenness": betweenness.get(node, 0),
                "closeness": closeness.get(node, 0),
                "eigenvector": eigenvector.get(node, 0),
                "pagerank": pagerank.get(node, 0),
                "katz": katz.get(node, 0)
            })
    except Exception as e:
        print(f"Error on {country}-{year}: {e}")
        continue

cent_df = pd.DataFrame(records)
print(f"\nCentrality records created: {len(cent_df)}")
print(cent_df.head(10).to_string(index=False))

cent_df.to_pickle(OUT_DATA / "centrality_measures.pkl")
cent_df.to_csv(OUT_TAB / "centrality_measures.csv", index=False)
print(f"\nSaved → {OUT_DATA / 'centrality_measures.pkl'}")

print("\nCell 2 completed successfully.")

Phase 5 – Network Analysis | Cell 2
Computing Centrality Measures

Centrality records created: 13590
  country  year          node  in_strength  out_strength  betweenness  closeness  eigenvector  pagerank      katz
Australia  1870    Government    17.256802      0.000002     0.416667   0.097911          0.0  0.054722  0.787393
Australia  1870   CentralBank     0.000000     17.799800     0.000000   0.000000          0.0  0.030000  0.288880
Australia  1870 BankingSector     0.543000     56.472000     0.000000   0.460405          0.0  0.030778  0.304566
Australia  1870      Industry   196.773601    131.165601     0.000000   0.097911          0.0  0.718362 -0.310386
Australia  1870     Household    45.496801     54.632801     0.166667   0.097911          0.0  0.166138  0.327811
Australia  1871    Government    19.179902      0.000002     0.416667   0.094239          0.0  0.054733  0.818787
Australia  1871   CentralBank     0.000000     19.774900     0.000000   0.000000          0.0  0.0300

In [3]:
# Phase 5 – Network Analysis
# Cell 3: Structural Metrics + Assortativity + Resilience Metrics

import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path
import pickle
from numpy.linalg import eigvals

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase5" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase5" / "tables"

with open(PROJECT / "outputs" / "phase4" / "data" / "temporal_networks_clean.pkl", "rb") as f:
    networks = pickle.load(f)

print("=" * 60)
print("Phase 5 – Network Analysis | Cell 3")
print("Structural + Resilience Metrics")
print("=" * 60)

records = []

for (country, year), G in networks.items():
    # Clean weights
    for u, v, d in G.edges(data=True):
        w = d.get("weight", 0)
        if w <= 0 or np.isnan(w):
            d["weight"] = 1e-6

    try:
        density = nx.density(G)
        
        # Diameter (on undirected version for simplicity)
        Gu = G.to_undirected()
        if nx.is_connected(Gu):
            diameter = nx.diameter(Gu)
        else:
            diameter = max([nx.diameter(Gu.subgraph(c)) for c in nx.connected_components(Gu) if len(c) > 1] or [0])
        
        # Clustering
        clustering = nx.average_clustering(Gu, weight="weight")
        
        # Assortativity (degree)
        try:
            assortativity = nx.degree_assortativity_coefficient(G, weight="weight")
        except:
            assortativity = np.nan
        
        # Global Efficiency
        global_eff = nx.global_efficiency(Gu)
        
        # Local Efficiency
        local_eff = nx.local_efficiency(Gu)
        
        # Spectral Gap & Algebraic Connectivity (from Laplacian)
        try:
            L = nx.normalized_laplacian_matrix(Gu).todense()
            eigenvalues = np.sort(np.real(eigvals(L)))
            algebraic_conn = eigenvalues[1] if len(eigenvalues) > 1 else 0
            spectral_gap = eigenvalues[1] - eigenvalues[0] if len(eigenvalues) > 1 else 0
        except:
            algebraic_conn = np.nan
            spectral_gap = np.nan
        
        records.append({
            "country": country,
            "year": year,
            "density": density,
            "diameter": diameter,
            "clustering": clustering,
            "assortativity": assortativity,
            "global_efficiency": global_eff,
            "local_efficiency": local_eff,
            "algebraic_connectivity": algebraic_conn,
            "spectral_gap": spectral_gap
        })
    except Exception as e:
        print(f"Error on {country}-{year}: {e}")
        continue

struct_df = pd.DataFrame(records)
print(f"\nStructural records: {len(struct_df)}")
print(struct_df.head(10).to_string(index=False))

struct_df.to_pickle(OUT_DATA / "structural_resilience_metrics.pkl")
struct_df.to_csv(OUT_TAB / "structural_resilience_metrics.csv", index=False)
print(f"\nSaved → {OUT_DATA / 'structural_resilience_metrics.pkl'}")

print("\nCell 3 completed successfully.")

Phase 5 – Network Analysis | Cell 3
Structural + Resilience Metrics

Structural records: 2718
  country  year  density  diameter  clustering  assortativity  global_efficiency  local_efficiency  algebraic_connectivity  spectral_gap
Australia  1870     0.55         2    0.063818       0.044011               0.85          0.483333                0.016880      0.016880
Australia  1871     0.55         2    0.064983       0.037740               0.85          0.483333                0.016874      0.016874
Australia  1872     0.55         2    0.057226       0.042144               0.85          0.483333                0.023001      0.023001
Australia  1873     0.55         2    0.065139       0.058354               0.85          0.483333                0.026556      0.026556
Australia  1874     0.55         2    0.063462       0.048881               0.85          0.483333                0.021460      0.021460
Australia  1875     0.55         2    0.062970       0.046033               0.85    

In [4]:
# Phase 5 – Network Analysis
# Cell 4: Build Final Feature Matrix

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase5" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase5" / "tables"

print("=" * 60)
print("Phase 5 – Network Analysis | Cell 4")
print("Building Feature Matrix")
print("=" * 60)

# Load previous results
cent = pd.read_pickle(OUT_DATA / "centrality_measures.pkl")
struct = pd.read_pickle(OUT_DATA / "structural_resilience_metrics.pkl")

# Pivot centrality to wide format (one row per country-year)
cent_wide = cent.pivot_table(
    index=["country", "year"],
    columns="node",
    values=["in_strength", "out_strength", "betweenness", "closeness", 
            "eigenvector", "pagerank", "katz"],
    aggfunc="first"
)

# Flatten column names
cent_wide.columns = [f"{metric}_{node}" for metric, node in cent_wide.columns]
cent_wide = cent_wide.reset_index()

# Merge with structural metrics
feature_matrix = pd.merge(cent_wide, struct, on=["country", "year"], how="left")

print(f"Feature Matrix shape: {feature_matrix.shape}")
print(f"Columns ({len(feature_matrix.columns)}):")
print(feature_matrix.columns.tolist())

print("\n--- Sample ---")
print(feature_matrix.head(3).to_string(index=False))

# Save
feature_matrix.to_pickle(OUT_DATA / "feature_matrix.pkl")
feature_matrix.to_csv(OUT_TAB / "feature_matrix.csv", index=False)

print(f"\nSaved Feature Matrix → {OUT_DATA / 'feature_matrix.pkl'}")
print(f"Saved Feature Matrix → {OUT_TAB / 'feature_matrix.csv'}")

print("\nCell 4 completed successfully.")

Phase 5 – Network Analysis | Cell 4
Building Feature Matrix
Feature Matrix shape: (2718, 45)
Columns (45):
['country', 'year', 'betweenness_BankingSector', 'betweenness_CentralBank', 'betweenness_Government', 'betweenness_Household', 'betweenness_Industry', 'closeness_BankingSector', 'closeness_CentralBank', 'closeness_Government', 'closeness_Household', 'closeness_Industry', 'eigenvector_BankingSector', 'eigenvector_CentralBank', 'eigenvector_Government', 'eigenvector_Household', 'eigenvector_Industry', 'in_strength_BankingSector', 'in_strength_CentralBank', 'in_strength_Government', 'in_strength_Household', 'in_strength_Industry', 'katz_BankingSector', 'katz_CentralBank', 'katz_Government', 'katz_Household', 'katz_Industry', 'out_strength_BankingSector', 'out_strength_CentralBank', 'out_strength_Government', 'out_strength_Household', 'out_strength_Industry', 'pagerank_BankingSector', 'pagerank_CentralBank', 'pagerank_Government', 'pagerank_Household', 'pagerank_Industry', 'density', 